# PortfolioOS Benchmarked: research case study

**SYNTHETIC DATA - mechanics demonstration, not empirical alpha evidence.**

Can transparent cross-sectional signals retain useful benchmark-relative performance after active-risk constraints, turnover limits and transaction costs are imposed?

This presentation calls package functions; it contains no duplicate portfolio logic. Underperformance is a valid result. Install the package and use its Python environment as the notebook kernel. Run from the repository root or notebooks directory.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

from portfolioos.reporting import load_config, run_demo

root = Path.cwd() if Path("configs/demo.yaml").exists() else Path.cwd().parent
config_path = root / "configs/demo.yaml"
synthetic_settings, config = load_config(config_path)
synthetic_settings, config

## Data and signals

The offline generator has 40 fixed assets, five sectors and independent market, sector and idiosyncratic shocks. Heterogeneous loadings and volatilities create correlations. Seed 42 is fixed. There is no embedded predictive signal structure, changing membership or exchange calendar.

Prices must be positive, chronological and unique. Missing data is rejected by default; only explicitly bounded causal forward fill is available. No future-based universe selection is performed.

For information date s=t-1, momentum is P[s-21]/P[s-252]-1, low volatility is negative trailing 63-return sample standard deviation, and reversal is -(P[s]/P[s-21]-1). Each cross section is winsorized at 5%/95%, z-scored and combined with fixed coefficients 1, 1, 0.5. These are price-based signals, not fundamental value or quality.

## Portfolio construction and timing

Ledoit-Wolf covariance uses 252 returns through t-1, adds a daily 1e-10 ridge and annualizes by 252. The optimizer maximizes score exposure minus annual active variance times 10 and full L1 turnover times 0.1. Scores are preferences, not calibrated expected returns.

Targets sum to one, are long-only, and satisfy 8% position, 8% annual estimated tracking error, 10 percentage point sector active and 30% one-way turnover caps. CLARABEL failures are explicit; no constraints are silently relaxed.

The first rebalance follows 253 price observations. Subsequent rebalances are monthly. An exclusive historical slice ends at t-1. Targets earn return t and then drift. This idealizes previous-close execution after observing that close; real execution requires additional lag or intraday data.

The equal-weight benchmark rebalances alongside the strategy and otherwise drifts. It is not a named market index. Supplied benchmark target snapshots instead update the benchmark on the next trading date after publication.

## Turnover and transaction costs

Initial capital is endowed in benchmark holdings. One-way turnover is half the absolute weight change from drifted pre-trade holdings, including the initial active transition. A complete switch has turnover one.

Cost = turnover * 10/10,000 in the default demo. Net return = gross return - cost, deducted once. This is an additive proportional NAV charge; relative asset holdings drift using gross returns. The frictionless benchmark incurs no trading costs. Realistic cash, lot sizes, liquidity and market impact are not modelled.

In [ ]:
summary = run_demo(config_path, root / "results/demo")
summary["metrics"]

## Metrics and diagnostics

Annualization uses 252 observations/year. CAGR compounds; volatility and tracking error use sample standard deviation. Sharpe uses zero risk-free rate by default. Sortino uses downside squared returns over all days. Drawdown includes initial NAV.

Net active return is portfolio minus benchmark return. Information ratio is mean daily active return *252 divided by annual tracking error. Undefined ratios are null. Cost drag distinguishes summed daily charges from compounded gross-minus-net wealth.

IC correlates rebalance scores with subsequent period buy-and-hold asset returns using Spearman ranks. It is computed after the simulation and never feeds construction. The terminal window may be partial; mean/std IC ratio is not annualized and is not an inference of significance.

## Attribution and generated charts

Security active contribution is beginning active weight * asset return. Daily contributions reconcile to gross active return; subtract cost for net active return.

Brinson-Fachler allocation is (Wp-Wb)(Rb-RB), selection Wb(Rp-Rb), interaction (Wp-Wb)(Rp-Rb). Their sum reconciles daily. An absent side's sector return is zero, making individual effects convention-dependent while preserving total reconciliation. Daily effects are not compounded multi-period attribution.

The generated report provides eight charts and the metric table. Image links below use notebook-relative paths.

In [ ]:
report_text = (root / "results/demo/report.md").read_text(encoding="utf-8")
image_prefix = "../results/demo/" if Path.cwd().name == "notebooks" else "results/demo/"
for filename in (root / "results/demo").glob("*.png"):
    report_text = report_text.replace(
        f"]({filename.name})", f"]({image_prefix}{filename.name})"
    )
display(Markdown(report_text))

## Substitute real historical data

Use pandas to read adjusted prices indexed by date, benchmark target snapshots indexed by their availability dates, and metadata indexed by ticker with a sector column. Call `run_backtest(prices, benchmark_weights, metadata, config=config)` and `write_report` with truthful provenance. `scripts/run_research.py` provides the equivalent CSV interface; no provider is required.

Optional external signals use the same dated matrix layout and need an explicit `external` coefficient. Omit metadata only after disabling the sector constraint. Benchmark constituents unavailable in prices are rejected. Do not commit licensed provider inputs.

## Limitations and interpretation

The fixed universe, constant factor distributions, idealized close timing, simplified costs, static sectors and absence of delistings limit real-world conclusions. Target limits hold at rebalances; drift and realized tracking error can exceed them. Dependencies and solvers can produce small numerical differences across platforms.

Further empirical work needs point-in-time membership and adjustments, execution lags, cost sensitivity, independent holdouts, regime analysis and proper inference. No real performance percentages, statistical significance or persistent alpha are fabricated. This case study demonstrates a reproducible research process.